In [8]:
import requests as res
import pandas as pd 

with open("./config/wthrkey", "r") as fp:
    API_KEY = fp.read().strip()

CITY = "Kochi"
URL = f"http://api.openweathermap.org/data/2.5/weather?q={CITY}&appid={KEY}&units=metric"
print(res.get(URL).json())

{'coord': {'lon': 76.2602, 'lat': 9.9399}, 'weather': [{'id': 802, 'main': 'Clouds', 'description': 'scattered clouds', 'icon': '03d'}], 'base': 'stations', 'main': {'temp': 25.28, 'feels_like': 25.98, 'temp_min': 25.28, 'temp_max': 25.28, 'pressure': 1012, 'humidity': 81, 'sea_level': 1012, 'grnd_level': 1012}, 'visibility': 10000, 'wind': {'speed': 2.24, 'deg': 37, 'gust': 2.4}, 'clouds': {'all': 33}, 'dt': 1772420078, 'sys': {'country': 'IN', 'sunrise': 1772413747, 'sunset': 1772456736}, 'timezone': 19800, 'id': 1273874, 'name': 'Kochi', 'cod': 200}


In [ ]:
import requests
import pandas as pd
import numpy as np
import tkinter as tk
from tkinter import messagebox
from matplotlib.figure import Figure
from matplotlib.backends.backend_tkagg import FigureCanvasTkAgg

with open("./config/wthrkey", "r") as fp:
    API_KEY = fp.read().strip()

class WeatherAnalyzerApp:
    def __init__(self, root):
        self.root = root
        self.root.title("Weather Data Analyzer")
        self.root.geometry("900x700")

        # --- UI: Search Bar ---
        search_frame = tk.Frame(self.root)
        search_frame.pack(pady=10)
        
        tk.Label(search_frame, text="Enter City:").pack(side=tk.LEFT)
        self.city_entry = tk.Entry(search_frame, width=30)
        self.city_entry.pack(side=tk.LEFT, padx=5)
        
        fetch_btn = tk.Button(search_frame, text="Fetch 5-Day Forecast", command=self.fetch_and_analyze)
        fetch_btn.pack(side=tk.LEFT)

        # --- UI: Plot Area ---
        self.fig = Figure(figsize=(8, 5), dpi=100)
        self.canvas = FigureCanvasTkAgg(self.fig, master=self.root)
        self.canvas.get_tk_widget().pack(fill=tk.BOTH, expand=True)

    def fetch_and_analyze(self):
        city = self.city_entry.get()
        if not city:
            messagebox.showwarning("Input Error", "Please enter a city name.")
            return

        # 1. Fetch 5-Day Forecast
        url = f"http://api.openweathermap.org/data/2.5/forecast?q={city}&appid={API_KEY}&units=metric"
        res = requests.get(url)
        
        if res.status_code != 200:
            messagebox.showerror("API Error", f"Error: {res.json().get('message', 'Unknown error')}")
            return

        raw_data = res.json()
        forecast_list = []
        for entry in raw_data['list']:
            forecast_list.append({
                "Date": entry['dt_txt'],
                "Temperature": entry['main']['temp'],
                "Humidity": entry['main']['humidity'],
                "Wind Speed": entry['wind']['speed'],
                "Weather Condition": entry['weather'][0]['main']
            })
        
        df = pd.DataFrame(forecast_list)
        df['Date'] = pd.to_datetime(df['Date'])

        # 2. Simulated Cleaning
        # manually add a duplicate and a null to show cleaning logic works
        df = pd.concat([df, df.iloc[[0]]]) #concatenate copy of first row to the end
        df.loc[1, 'Temperature'] = np.nan
        df.to_csv("./datasets/raw_wthr.csv") # export raw weather data to csv (backup)
        
        # Handle missing values and duplicates
        df['Temperature'] = df['Temperature'].fillna(df['Temperature'].mean())
        df['Humidity'] = df['Humidity'].fillna(df['Humidity'].median())
        df = df.drop_duplicates()

        self.plot_trends(df, city)

    def plot_trends(self, df, city):
        self.fig.clear()
        
        # Plot Temperature Trends
        ax1 = self.fig.add_subplot(211)
        ax1.plot(df['Date'], df['Temperature'], color='red', marker='o', linestyle='-')
        ax1.set_title(f"5-Day Temperature Trend for {city}")
        ax1.set_ylabel("Temp (°C)")
        ax1.grid(True)

        # Plot Temp vs Humidity
        ax2 = self.fig.add_subplot(212)
        ax2.scatter(df['Temperature'], df['Humidity'], alpha=0.5, color='blue')
        ax2.set_title("Relationship: Temp vs Humidity")
        ax2.set_xlabel("Temperature (°C)")
        ax2.set_ylabel("Humidity (%)")
        ax2.grid(True)

        self.fig.tight_layout()
        self.canvas.draw()

if __name__ == "__main__":
    root = tk.Tk()
    app = WeatherAnalyzerApp(root)
    root.mainloop()

../../../../modules/im/ximcp/imDefLkup.c,355: The key event is already fabricated.
